# Capstone 2: Automated BRD Analyzer

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build an **Automated BRD Analyzer** that takes a Business Requirements Document, checks completeness against a template, identifies gaps, scores quality, extracts key entities, and generates actionable review comments.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Completed Chapters 1-15


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Sample BRD Input

A realistic (but intentionally incomplete) BRD for analysis.


In [ ]:
# Sample BRD content (intentionally incomplete for the analyzer to catch)
brd_content = """
BUSINESS REQUIREMENTS DOCUMENT
Project: Online Learning Platform v2.0
Author: Jane Smith, Senior BA
Date: 2024-11-01
Version: 1.2

1. EXECUTIVE SUMMARY
The Online Learning Platform v2.0 will add live virtual classrooms, 
AI-powered learning paths, and a mobile app to the existing web-based 
course catalog system. This upgrade aims to increase user engagement 
by 40% and reduce course dropout rates.

2. BUSINESS OBJECTIVES
- Increase monthly active users from 50K to 80K within 6 months of launch
- Reduce course dropout rate from 35% to 20%
- Generate $2M additional annual revenue from premium features
- Achieve NPS score of 45+

3. SCOPE
In Scope:
- Live virtual classroom with video, screen sharing, and chat
- AI-generated personalized learning paths
- Mobile app for iOS and Android
- Integration with existing LMS
- Certificate generation

Out of Scope:
- Content creation tools
- Payment gateway changes

4. FUNCTIONAL REQUIREMENTS
FR-001: The system shall support live virtual classrooms with up to 100 participants.
FR-002: The system shall use AI to recommend courses based on user history and goals.
FR-003: Users shall be able to download course materials for offline access.
FR-004: The system shall generate certificates upon course completion.
FR-005: Instructors shall be able to create and schedule live sessions.

5. NON-FUNCTIONAL REQUIREMENTS
NFR-001: The platform should be fast.
NFR-002: The system shall support 10,000 concurrent users.
NFR-003: All data shall be encrypted.

6. ASSUMPTIONS
- Users have stable internet connections
- Existing user accounts will be migrated automatically
"""

print(f'BRD loaded: {len(brd_content)} characters')

## 2. BRD Template Completeness Check

Compare the BRD against a standard template to identify missing sections.


In [ ]:
# BRD template sections
brd_template = [
    'Executive Summary', 'Business Objectives', 'Scope (In/Out)',
    'Stakeholder Analysis', 'Current State Analysis', 'Future State Description',
    'Functional Requirements', 'Non-Functional Requirements',
    'Data Requirements', 'Integration Requirements',
    'User Interface Requirements', 'Reporting Requirements',
    'Assumptions', 'Constraints', 'Dependencies',
    'Risks and Mitigations', 'Success Metrics / KPIs',
    'Timeline / Milestones', 'Budget Estimate',
    'Glossary', 'Approval Sign-off'
]

completeness_prompt = f"""Analyze this BRD for completeness against the required template sections.

BRD Content:
{brd_content}

Required Sections: {json.dumps(brd_template)}

For each required section:
- present: true/false
- quality: if present, rate 1-5 (1=stub, 5=comprehensive)
- gaps: what's missing or weak
- recommendation: what should be added

Also provide:
- completeness_score: percentage of sections adequately covered
- critical_gaps: sections that MUST be added before approval

Return as JSON. Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': completeness_prompt}], temperature=0.2
)
completeness = json.loads(response.choices[0].message.content)
print(f"BRD Completeness: {completeness['completeness_score']}%\n")
print('Missing or incomplete sections:')
for section in completeness.get('sections', completeness.get('required_sections', [])):
    if isinstance(section, dict) and not section.get('present', True):
        print(f"  [MISSING] {section.get('section', section.get('name', 'N/A'))}")
    elif isinstance(section, dict) and section.get('quality', 5) < 3:
        print(f"  [WEAK]    {section.get('section', section.get('name', 'N/A'))}: {section.get('gaps', '')}")

## 3. Requirements Quality Analysis

Deep-dive into the quality of individual requirements.


In [ ]:
# Analyze individual requirements for quality
req_quality_prompt = f"""Analyze every requirement (FR and NFR) in this BRD for quality.

BRD Content:
{brd_content}

For each requirement, assess:
- id: requirement ID
- text: the requirement text
- ambiguity_score: 1-5 (5=perfectly clear)
- testability_score: 1-5 (5=easily testable)
- completeness_score: 1-5 (5=fully complete)
- issues: list of specific problems
- improved_version: rewritten requirement

Return a JSON object with:
- requirements: array of assessments
- average_quality: overall average score
- worst_requirements: IDs of the bottom 2

Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': req_quality_prompt}], temperature=0.2
)
req_quality = json.loads(response.choices[0].message.content)
print(f"Average Quality: {req_quality['average_quality']}/5")
print(f"Worst Requirements: {req_quality['worst_requirements']}\n")
for req in req_quality['requirements']:
    avg = (req['ambiguity_score'] + req['testability_score'] + req['completeness_score']) / 3
    print(f"{req['id']}: Avg={avg:.1f} | Ambiguity={req['ambiguity_score']} Testability={req['testability_score']} Completeness={req['completeness_score']}")
    if req.get('issues'):
        for issue in req['issues'][:2]:
            print(f"  Issue: {issue}")

In [ ]:
# Generate a comprehensive review report
review_prompt = f"""Generate a formal BRD review report with actionable feedback.

BRD Content:
{brd_content}

Completeness Analysis: {json.dumps(completeness)}
Requirements Quality: {json.dumps(req_quality)}

The review report should include:
1. Review Summary (pass/fail/conditional)
2. Completeness Findings (missing sections with priority)
3. Requirements Quality Findings (specific issues per requirement)
4. Consistency Check (contradictions or conflicts between sections)
5. Risk Assessment (risks from gaps in the BRD)
6. Required Actions (numbered list of changes needed before approval)
7. Reviewer Recommendation

Format as a professional review document."""

response = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': review_prompt}], temperature=0.3
)
print('BRD REVIEW REPORT')
print('=' * 60)
print(response.choices[0].message.content)

## Exercises
1. Upload a real BRD from your organization and run the analyzer on it.
2. Add entity extraction that identifies all actors, systems, data objects, and business rules.
3. Create a BRD comparison tool that diffs two versions and highlights changes.
4. Build a BRD completion assistant that generates content for missing sections based on existing context.
